In [ ]:
import socket

print(socket.gethostname())


midway3-0055.rcc.local


In [ ]:
import json
import pandas as pd

with open(
    "/project/rcc/users/hdashti/projects/shiplogs/oldweather/geo/manifest.json"
) as f:
    m = json.load(f)

ships_df = pd.DataFrame(m["ships"])
print(f"Ships in manifest: {len(ships_df)}")
print(f"\nTotal points across all ships: {ships_df['n_points'].sum():,}")
print(f"Total track segments:          {ships_df['n_track_segments'].sum():,}")
print(f"Total track distance:          {ships_df['total_track_nm'].sum():,.0f} nm")

print(f"\nShips with zero track segments:")
zero_trk = ships_df[ships_df["n_track_segments"] == 0]
print(
    zero_trk[["ship", "n_points", "n_anchored", "n_underway"]]
    if len(zero_trk)
    else "  (none)"
)

print(f"\nShips with rows missing coords:")
missing = ships_df[ships_df["n_dropped_no_coords"] > 0]
print(
    missing[["ship", "n_points", "n_dropped_no_coords"]] if len(missing) else "  (none)"
)

print(f"\nBiggest GeoJSON files (MB):")
ships_df["pts_geojson_mb"] = ships_df["pts_geojson_kb"] / 1024
print(
    ships_df.nlargest(10, "pts_geojson_mb")[
        ["ship", "n_points", "pts_geojson_mb"]
    ].to_string(index=False)
)

print(f"\nSpatial bounds overall:")
b = pd.DataFrame(ships_df["bounds"].tolist(), columns=["W", "S", "E", "N"])
print(f"  lon: {b['W'].min():.2f} to {b['E'].max():.2f}")
print(f"  lat: {b['S'].min():.2f} to {b['N'].max():.2f}")

print(
    f"\nDate range overall: {ships_df['start_date'].min()} to {ships_df['end_date'].max()}"
)

Ships in manifest: 42

Total points across all ships: 2,043,536
Total track segments:          14,885
Total track distance:          3,891,243 nm

Ships with zero track segments:
         ship  n_points  n_anchored  n_underway
5   BlackHawk       720         720           0
30       Rush      1148           0        1148

Ships with rows missing coords:
             ship  n_points  n_dropped_no_coords
3            Atka      2626                   69
7   Burton_Island     77446                    1
14      Jeannette     16890                  318
15      Kearsarge    141013                   44
16     Lackawanna    107062                    1
17      Lancaster     86804                  124
20        Mohican     29517                    3
22   Narragansett     17831                    1
25          Omaha    102838                    3
31     Sacramento     34727                    1
33     Shenandoah    128423                    2
38    Ticonderoga     36070                    2
39     

In [ ]:
import pandas as pd
import numpy as np
import json
import os

SRC = "/project/rcc/users/hdashti/projects/shiplogs/oldweather/oldweather_clean/oldweather_cleaned_dedup.parquet"
OUT = "/home/hdashti/projects/shiplog/source/kepler_trips_1900.geojson"

df = pd.read_parquet(SRC)
y = df[(df["datetime"].dt.year == 1900)].copy()
y = (
    y.dropna(subset=["lat", "lon"])
    .sort_values(["ship", "datetime"])
    .reset_index(drop=True)
)

print(f"1900 rows with coords: {len(y):,}")
print(f"Ships in 1900: {y['ship'].nunique()}")
print(y["ship"].value_counts())

y["ts"] = (y["datetime"].astype("int64") // 10**9).astype(int)
y["temp_water"] = y["temp_water"].where(y["temp_water"].notna(), None)

features = []
for ship, g in y.groupby("ship"):
    coords = g[["lon", "lat"]].values
    ts = g["ts"].values
    path = [[float(lon), float(lat), 0, int(t)] for (lon, lat), t in zip(coords, ts)]

    temps = g["temp_water"].dropna()
    features.append(
        {
            "type": "Feature",
            "properties": {
                "ship": ship,
                "n_points": int(len(g)),
                "start": g["datetime"].min().strftime("%Y-%m-%dT%H:%M:%S"),
                "end": g["datetime"].max().strftime("%Y-%m-%dT%H:%M:%S"),
                "mean_sea_temp_F": round(float(temps.mean()), 2)
                if len(temps)
                else None,
            },
            "geometry": {"type": "LineString", "coordinates": path},
        }
    )

fc = {"type": "FeatureCollection", "features": features}

with open(OUT, "w") as f:
    json.dump(fc, f)

print(f"\nWrote {OUT}  ({os.path.getsize(OUT) / 1024 / 1024:.1f} MB)")
print(f"Features: {len(features)}")

1900 rows with coords: 12,336
Ships in 1900: 4
ship
Bear         5112
McCulloch    3671
Manning      3408
Rush          145
Name: count, dtype: int64

Wrote /home/hdashti/projects/shiplog/source/kepler_trips_1900.geojson  (0.4 MB)
Features: 4


In [ ]:
import json

OUT = "/home/hdashti/projects/shiplog/source/test.geojson"

data = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "properties": {"vendor": "A", "vol": 20},
            "geometry": {
                "type": "LineString",
                "coordinates": [
                    [-74.20986, 40.81773, 0, 1564184363],
                    [-74.20987, 40.81765, 0, 1564184396],
                    [-74.20998, 40.81746, 0, 1564184409],
                ],
            },
        }
    ],
}

with open(OUT, "w") as f:
    json.dump(data, f, indent=2)

print(f"Wrote {OUT}")

1900 rows with coords: 12,336
Ships in 1900: 4

Wrote /home/hdashti/projects/shiplog/source/kepler_trips_1900_v2.geojson
Size: 0.4 MB
Features: 4

First feature (truncated):
{
  "type": "Feature",
  "properties": {
    "ship": "Bear"
  },
  "geometry": {
    "type": "LineString",
    "coordinates": [
      [
        -122.49,
        37.86,
        0,
        -2199052800
      ],
      [
        -122.49,
        37.86,
        0,
        -2199049200
      ],
      [
        -122.49,
        37.86,
        0,
        -2199045600
      ],
      "..."
    ]
  }
}
